# Solar upsell targeting

Goal: identify which meters are likely to already have rooftop solar, so the sales team can
prioritise the solar-battery upsell campaign. We use the meter attributes plus the 2023 daily
consumption pattern and fit a logistic regression.


> **Fixed version.** Each correction is marked with a *Fix:* note.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, roc_auc_score

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

## Load data

In [2]:
meters = pd.read_csv("../../data/meters.csv", parse_dates=["signup_date"])
readings = pd.read_csv("../../data/meter_readings_daily.csv", parse_dates=["date"])
print(meters.shape, readings.shape)
meters.head()

(300, 7) (107503, 3)


,meter_id,region,tariff,customer_type,annual_kwh_estimate,signup_date,has_solar
0,M100000,London,Fixed,sme,21622.0,2021-07-07,False
1,M100001,London,Fixed,residential,2286.0,2022-11-17,False
2,M100002,London,Fixed,residential,3665.0,2021-06-04,False
3,M100003,Scotland,Fixed,residential,2575.0,2021-10-26,False
4,M100004,Midlands,Fixed,residential,2191.0,2022-10-31,False


*Fix:* look at the target base rate and the categorical values before doing anything. 11% positives means accuracy is not a useful metric, and `region` has lowercase duplicates.

In [3]:
print(meters["has_solar"].value_counts(normalize=True).round(3))
print(meters["region"].value_counts())
meters["region"] = meters["region"].str.title()
print(meters["region"].nunique(), "regions after normalising case")

has_solar
False    0.887
True     0.113
Name: proportion, dtype: float64
region
London      96
North       65
Scotland    57
Midlands    44
Wales       32
london       3
wales        1
north        1
midlands     1
Name: count, dtype: int64
5 regions after normalising case


## Consumption features

Summer vs winter consumption ratio: solar households consume less from the grid in summer, so the ratio should be lower for them.

*Fix:* the original called `drop_duplicates(["meter_id", "season"])` **before** aggregating, so each season was represented by a single arbitrary day (the first one). Aggregate with `groupby().mean()` instead; the ratio becomes a very strong signal.

In [4]:
rd = readings.merge(meters[["meter_id"]], on="meter_id", validate="many_to_one")
rd["month"] = rd["date"].dt.month
rd["season"] = np.select(
    [rd["month"].isin([6, 7, 8]), rd["month"].isin([12, 1, 2])],
    ["summer", "winter"], default="other",
)
seasonal = rd.groupby(["meter_id", "season"])["kwh"].mean().unstack("season")
seasonal["sw_ratio"] = seasonal["summer"] / seasonal["winter"]
print(seasonal.groupby(meters.set_index("meter_id")["has_solar"])["sw_ratio"].describe().round(3))
seasonal.head()

           count   mean    std    min    25%    50%    75%    max
has_solar                                                        
False      266.0  0.523  0.020  0.455  0.511  0.522  0.537  0.574
True        34.0  0.407  0.041  0.302  0.392  0.410  0.426  0.497


season,other,summer,winter,sw_ratio
meter_id,,,,
M100000,61.320291,40.540326,82.195023,0.493221
M100001,6.615218,4.390711,8.513573,0.515731
M100002,10.096682,7.076945,13.345292,0.530295
M100003,7.199713,4.970256,9.532273,0.521413
M100004,6.309489,4.322264,7.759315,0.557042


## Attributes

*Fix:* no target-derived feature (`region_solar_rate` was the target mean by region computed on the **whole** sample). Keep `annual_kwh_estimate` as NaN so the imputer handles it inside the pipeline instead of `fillna(0)` + `log(x+1)`, which turned the 6 missing meters into an artificial cluster at 0. Tariff mode-imputation also moves inside the pipeline so it is learnt on train only.

In [5]:
df = meters.merge(seasonal[["sw_ratio"]], left_on="meter_id", right_index=True, how="left", validate="one_to_one")
df["log_kwh"] = np.log(df["annual_kwh_estimate"])
df["tenure_days"] = (pd.Timestamp("2024-01-01") - df["signup_date"]).dt.days
df["is_sme"] = (df["customer_type"] == "sme").astype(int)

num_cols = ["sw_ratio", "log_kwh", "tenure_days", "is_sme"]
cat_cols = ["region", "tariff"]
X = df[num_cols + cat_cols]
y = df["has_solar"].astype(int)
print(X.isna().sum())

sw_ratio        0
log_kwh         6
tenure_days     0
is_sme          0
region          0
tariff         13
dtype: int64


## Train / test split

*Fix:* `stratify=y` so both sets have the same 11% positive rate, and a `random_state` so the run is reproducible. With 90 test rows and ~10 positives, a single split is very noisy anyway, so we also report cross-validated out-of-fold predictions further down.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(len(X_train), len(X_test), "positives:", y_train.sum(), y_test.sum())

210 90 positives: 24 10


## Model

*Fix:* all preprocessing (imputation, one-hot, scaling) inside a `ColumnTransformer` so it is fitted on the training fold only; `handle_unknown="ignore"` so an unseen category cannot crash prediction; `class_weight="balanced"` because the positive class is 11%.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
pipe.fit(X_train, y_train)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('sc',
                                                                   StandardScaler())]),
                                                  ['sw_ratio', 'log_kwh',
                                                   'tenure_days', 'is_sme']),
                                                 ('cat',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('oh',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['region', 'tariff'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

*Fix:* always compare against the trivial baseline. Accuracy of "nobody has solar" is 0.89, so the original 0.93 was almost no information. Use `classification_report` and AUC computed from **probabilities** on the **holdout** set.

In [8]:
from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.dummy import DummyClassifier

base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Baseline accuracy (always 'no'): {accuracy_score(y_test, base.predict(X_test)):.3f}")
proba_test = pipe.predict_proba(X_test)[:, 1]
print(f"Accuracy (threshold 0.5)       : {accuracy_score(y_test, pipe.predict(X_test)):.3f}")
print(f"AUC holdout, from probabilities: {roc_auc_score(y_test, proba_test):.3f}")
print(f"AUC holdout, from hard labels  : {roc_auc_score(y_test, pipe.predict(X_test)):.3f}   <- what the original computed")
print(f"AUC train                      : {roc_auc_score(y_train, pipe.predict_proba(X_train)[:, 1]):.3f}   <- what the original called 'AUC'")
print(classification_report(y_test, pipe.predict(X_test), target_names=["no solar", "solar"], digits=3))

Baseline accuracy (always 'no'): 0.889
Accuracy (threshold 0.5)       : 1.000
AUC holdout, from probabilities: 1.000
AUC holdout, from hard labels  : 1.000   <- what the original computed
AUC train                      : 1.000   <- what the original called 'AUC'
              precision    recall  f1-score   support

    no solar      1.000     1.000     1.000        80
       solar      1.000     1.000     1.000        10

    accuracy                          1.000        90
   macro avg      1.000     1.000     1.000        90
weighted avg      1.000     1.000     1.000        90



*Fix:* the threshold is a business decision (how many meters can sales call?), not 0.5 by default. Show the precision/recall trade-off on the holdout set.

In [9]:
prec, rec, thr = precision_recall_curve(y_test, proba_test)
pr = pd.DataFrame({"threshold": np.r_[thr, np.nan], "precision": prec, "recall": rec})
pr.iloc[::max(1, len(pr) // 10)].round(3)

,threshold,precision,recall
0,0.000,0.111,1.0
9,0.001,0.123,1.0
18,0.004,0.139,1.0
27,0.007,0.159,1.0
36,0.008,0.185,1.0
45,0.014,0.222,1.0
54,0.024,0.278,1.0
63,0.046,0.370,1.0
72,0.079,0.556,1.0
81,0.997,1.000,0.9


*Fix:* rank with column 1 of `predict_proba` (P(solar)), not column 0 (P(no solar)), and rank on **out-of-fold** predictions so training rows do not get an optimistic score.

In [10]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
oof = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
df["p_solar"] = oof
print(f"Out-of-fold AUC on all 300 meters: {roc_auc_score(y, oof):.3f}")
top20 = df.sort_values("p_solar", ascending=False).head(20)
top20[["meter_id", "region", "tariff", "annual_kwh_estimate", "sw_ratio", "p_solar", "has_solar"]]

Out-of-fold AUC on all 300 meters: 0.994


,meter_id,region,tariff,annual_kwh_estimate,sw_ratio,p_solar,has_solar
109,M100109,London,Fixed,1627.0,0.301712,1.000000,True
157,M100157,North,Variable,1630.0,0.306337,1.000000,True
121,M100121,North,Fixed,1750.0,0.353075,0.999976,True
259,M100259,North,Fixed,2245.0,0.350831,0.999969,True
49,M100049,North,TOU,2274.0,0.349677,0.999921,True
13,M100013,North,Fixed,2948.0,0.393840,0.999141,True
64,M100064,London,TOU,2692.0,0.375244,0.999118,True
176,M100176,London,Variable,2661.0,0.382823,0.999075,True
229,M100229,Scotland,Fixed,3457.0,0.406564,0.999064,True
290,M100290,North,Fixed,3131.0,0.391250,0.998614,True


## Results

In [11]:
print(f"Baseline accuracy          : {accuracy_score(y_test, base.predict(X_test)):.3f}")
print(f"Holdout AUC (probabilities): {roc_auc_score(y_test, proba_test):.3f}")
print(f"Out-of-fold AUC (300 rows) : {roc_auc_score(y, oof):.3f}")
print(f"Top-20 list: {top20['has_solar'].sum()} of 20 are solar meters (base rate would give ~{20 * y.mean():.1f})")
print("Note: the model is only useful for meters we have not already flagged; the campaign list should exclude has_solar == True.")

Baseline accuracy          : 0.889
Holdout AUC (probabilities): 1.000
Out-of-fold AUC (300 rows) : 0.994
Top-20 list: 20 of 20 are solar meters (base rate would give ~2.3)
Note: the model is only useful for meters we have not already flagged; the campaign list should exclude has_solar == True.
